[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/6_Synapses_and_Comm.ipynb)

# Set 6 — Synapses and Communication
**Author:** Nair SS, et al., Electrical Engineering and Computer Science, University of Missouri-Columbia, Columbia, MO, USA

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go. **E9 — Synaptic Playground** below is a good spot for an optional (AI-assisted) extension.

---
### 📋 Before You Begin: How This Module Is Graded
Within this module's combined Submissions + Quizzes score, the companion quiz is worth **66.7%** and the Colab submission is worth the remaining **33.3%**. Submissions + Quizzes together (across all 10 modules) make up **30%** of your overall course grade. The quiz specifically checks whether the understanding behind your submission is actually yours — pasting an AI's explanation or a friend's slider values can get this notebook looking complete, but it won't get you through a quiz question applying the same concept to a scenario you haven't seen before.

The **design problem** below, and the **two questions you'll create yourself**, both exist for exactly this reason.

---

<details>
<summary><b>📖 Click to expand: Synaptic Variable Key & Module Roadmap</b></summary>

### Variable Key: Synaptic Parameters
| Symbol | NEURON Name | Definition | Units |
| :--- | :--- | :--- | :--- |
| **$E_{rev}$** | `syn.e` | Reversal Potential (Determines Excitation vs. Inhibition) | mV |
| **$\tau$** | `syn.tau` | Decay Time Constant (Synaptic kinetics/speed) | ms |
| **$w$** | `nc.weight[0]` | Synaptic Weight (Strength of the connection) | μS (approx) |
| **$\Delta t$** | `nc.delay` | Axonal Latency (Time for signal to travel) | ms |
| **ISI** | `ns.interval` | Inter-Stimulus Interval (Time between presynaptic spikes) | ms |

---

### Conceptual Glossary
* **Driving Force:** The difference between $V_m$ and $E_{rev}$. It determines the direction and magnitude of current flow.
* **Temporal Summation:** When multiple inputs arrive at the same synapse in quick succession, building upon one another.
* **Spatial Summation:** The integration of inputs arriving at different physical locations (e.g., proximal vs. distal dendrites).
* **Shunting Inhibition:** Inhibition that occurs near the resting potential; it "divides" excitability by increasing conductance without changing voltage.
* **Feedforward Inhibition:** A circuit motif where an excitatory signal also triggers a delayed inhibitory signal to narrow the integration window.

---

### Module Roadmap
* **Biology Foundations** — the 6-step synapse mechanism + 5 synapse types
* **E0** — Starter: Environment Setup
* **E1** — Excitatory Logic (AMPA-like)
* **E2** — Inhibitory Logic (GABA_A-like)
* **E3** — Dynamics Contrast (EPSP vs. IPSP) + by-hand driving-force drill
* **E4** — Reversal Sweep (The Zero-Crossing)
* **E5** — Temporal Summation Logic
* **E6** — Spatial Summation & Dendritic Filtering
* **E7** — Feedforward Inhibition Motifs
* **E8** — Automated Signal Metrics
* **E9** — Synaptic Playground
* **Design Challenges** — rate coding, surgical spike suppression, location-dependent inhibition
* **Homework + Knowledge Bank**

</details>

## E0 — Starter: Environment Setup
> **Instruction:** In this section, we initialize the virtual laboratory. We establish a "Helper Function" (`mk_rec`) to act as our digital oscilloscope, allowing us to record voltage from any point on our virtual neuron.

### The Technical Environment
We use a standard **Hodgkin-Huxley (HH)** model for our sections. This ensures that even our "passive" experiments are grounded in a membrane that *could* spike if the stimulus is strong enough.
* **Resting Potential:** Calibrated to **-65mV**.
* **Recording:** We use `h.Vector()` to record both time (`t`) and voltage (`v`) at 0.1ms intervals.

In [1]:
#@title E0 — Starter: Environment Setup { display-mode: "form" }
!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt

h.load_file('stdrun.hoc')

# --- Global Helpers ---

def psp_metrics(tvec, vtrace, baseline=(0, 5)):
    """Extracts biophysical features from a voltage trace without deprecation warnings."""
    tnp = np.array(tvec)
    vnp = np.array(vtrace)

    # Ensure arrays match in size
    if tnp.size != vnp.size:
        min_size = min(tnp.size, vnp.size)
        tnp = tnp[:min_size]
        vnp = vnp[:min_size]

    b = (tnp >= baseline[0]) & (tnp <= baseline[1])
    v0 = vnp[b].mean() if b.any() else vnp[0]

    v_rel = vnp - v0
    idx_peak = np.argmax(np.abs(v_rel))
    peak_val = v_rel[idx_peak]
    t_peak = tnp[idx_peak]

    # FIX: Using np.trapezoid instead of the deprecated np.trapz
    area = np.trapezoid(v_rel, tnp)

    return {'V0': v0, 'Peak': peak_val, 't_peak': t_peak, 'Area': area}

# Helper to create recordings and manage global time
t = h.Vector()
def mk_rec(sec):
    # Wipe the global time vector to ensure synchronization with voltage
    if t.size() > 0:
        t.resize(0)
    t.record(h._ref_t)

    v = h.Vector()
    v.record(sec(0.5)._ref_v)
    return v

print('E0 environment and automated metrics ready.')

ERROR: Ignored the following yanked versions: 9.0a1, 9.0a2
ERROR: Could not find a version that satisfies the requirement neuron==8.2.4 (from versions: 8.2.7, 9.0rc0, 9.0rc1, 9.0.0, 9.0.1rc1, 9.0.1, 9.0.2)
ERROR: No matching distribution found for neuron==8.2.4


ModuleNotFoundError: No module named 'neuron'

## Biology Foundations: How a Chemical Synapse Actually Works
Before diving into the rigs, let's ground the mechanism in real biology — you'll be simulating this exact process.

**Activity — Ask AI:**

> *"Walk me through, step by step, what physically happens when an action potential reaches a presynaptic terminal and ends up causing (or not causing) a new action potential in the postsynaptic neuron. Include the role of voltage-gated calcium channels, synaptic vesicles, neurotransmitter diffusion, and receptor binding."*

Paste the response.
How a presynaptic action potential causes a postsynaptic response
The key idea is that an electrical signal in the presynaptic neuron is converted into a chemical signal at the synapse, then converted back into an electrical response in the postsynaptic neuron. The postsynaptic response may or may not be strong enough to trigger another action potential.
Step-by-step process



Outside cell
Resting membrane potential · 1/7
Inside cell
-70 mV


Na⁺K⁺VG Na⁺ channelVG K⁺ channelK⁺ leakNa⁺/K⁺ pump








Ready at rest
With voltage-gated channels closed, potassium leak helps set the resting voltage while the sodium-potassium pump maintains the ion gradients.










Give feedback










1. The action potential reaches the presynaptic terminal
An action potential travels along the axon and depolarizes the presynaptic terminal. The membrane voltage becomes more positive, activating voltage-gated calcium channels.












2. Voltage-gated Ca²⁺ channels open
Calcium ions enter the terminal because of their electrochemical gradient. The rise in intracellular calcium is the signal that triggers neurotransmitter release.
Unlike Na⁺ influx during the action potential upstroke, this calcium influx primarily initiates synaptic vesicle release.












3. Synaptic vesicles release neurotransmitter
Calcium activates the vesicle-fusion machinery. Synaptic vesicles fuse with the presynaptic membrane and release neurotransmitter into the synaptic cleft through exocytosis.












4. Neurotransmitter diffuses across the synaptic cleft
The molecules move across the tiny gap between the two cells and reach receptors on the postsynaptic membrane. Diffusion is rapid, but the amount of neurotransmitter released and its clearance affect the strength and duration of the signal.












5. Neurotransmitter binds to postsynaptic receptors
For an ionotropic receptor, binding directly opens an ion channel. Ions flow across the postsynaptic membrane, changing its voltage. Other receptors, called metabotropic receptors, act through intracellular signaling pathways and can change ion-channel activity indirectly.












6. The postsynaptic voltage changes
An excitatory postsynaptic potential (EPSP) generally moves the membrane toward firing threshold. An inhibitory postsynaptic potential (IPSP) generally makes firing less likely, through hyperpolarization, shunting, or both.












7. The postsynaptic neuron either fires or does not fire
If the combined inputs bring the axon initial segment to threshold, voltage-gated Na⁺ channels activate and a new action potential begins. If the membrane stays below threshold, the neuron produces a graded response without a full action potential.



The equations connecting these steps
These equations are especially useful for your two-neuron simulation.
1. Calcium entry and neurotransmitter release
A simplified relationship is
\[
I_{\mathrm{Ca}}=g_{\mathrm{Ca}}q(V_{\mathrm{pre}}-E_{\mathrm{Ca}})
\]

Here, \(q\) represents calcium-channel activation. Calcium entry increases intracellular calcium concentration, which promotes vesicle fusion and neurotransmitter release.
Release is nonlinear in calcium concentration; the equation above describes the calcium current, not the full vesicle-release process.
2. Synaptic activation
A simple first-order model of synaptic conductance is
\[
\boxed{\frac{ds}{dt}=\alpha T(V_{\mathrm{pre}})(1-s)-\beta s}
\]

where:
- \(s\) is the fraction of activated synaptic channels.
- \(T(V_{\mathrm{pre}})\) represents presynaptic activation.
- \(\alpha\) controls activation.
- \(\beta\) controls deactivation.
This is a simplified model: a more physically detailed version would make vesicle release depend on presynaptic calcium and neurotransmitter concentration.
3. Postsynaptic current
\[
\boxed{I_{\mathrm{syn}}=g_{\mathrm{syn}}s(V_{\mathrm{post}}-E_{\mathrm{syn}})}
\]

The current depends on the synaptic conductance, how many channels are activated, and the difference between the postsynaptic voltage and the synaptic reversal potential.
4. Postsynaptic voltage
For a passive postsynaptic neuron,
\[
\boxed{
C_m\frac{dV_{\mathrm{post}}}{dt}
=-g_L(V_{\mathrm{post}}-E_L)-I_{\mathrm{syn}}
+I_{\mathrm{ext}}
}
\]

This equation predicts the graded postsynaptic voltage response. To model a postsynaptic action potential, add voltage-gated Na⁺ and K⁺ currents, as in the Hodgkin–Huxley model.
What determines whether the postsynaptic neuron fires?
Postsynaptic firing

The EPSP is strong enough to bring the membrane to threshold. Voltage-gated Na⁺ channels activate, producing a regenerative action potential.

No postsynaptic firing

The EPSP is too small to reach threshold. The neuron may depolarize, but it does not produce a full action potential.

The result depends on neurotransmitter release, receptor properties, synaptic strength, membrane resistance, concurrent inhibitory inputs, and the postsynaptic neuron's threshold and recent activity.
Connection to your model: To make neuron 1 fire without neuron 2 firing, keep the presynaptic current sufficient to generate action potentials and lower g_syn until neuron 2 receives a measurable but subthreshold response. That models successful synaptic transmission without a postsynaptic spike.

*Your Answer Here — in your own words, why can't the electrical signal just "jump" across the synaptic cleft directly?*
It cannot because there is a capacitance to overcome between the two neurons.

### 📸 Upload Checkpoint 1 — The Six-Step Synapse Mechanism, From Memory
**Before you write anything: stop, close this notebook (or at minimum, don't scroll back up).** Without looking back at the AI response you pasted above, write out the six-step chemical synapse mechanism from memory, start to finish — action potential arrival through postsynaptic response.

Only after you've written what you can recall should you check it against your own earlier work in this notebook. Then photograph your hand-written page and upload it below.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file (`from google.colab import files`) and display it inline.

In [ ]:
# Your image upload + display code here:


### The Five Types of Synapses
Not every synapse works the same way. **Activity — Ask AI:**

> *"Briefly describe five different types of synapses found in the nervous system: excitatory ion channel synapses, inhibitory ion channel synapses, non-channel (second-messenger) synapses, neuromuscular junctions, and electrical synapses. For each, name a typical neurotransmitter or mechanism."*

Paste the response.

Five types of synapses
Synapse type	Description	Typical neurotransmitter or mechanism
1. Excitatory ion-channel synapse	Opens postsynaptic ion channels that generally depolarize the membrane, making an action potential more likely.	Glutamate activates AMPA receptors, allowing positive ions such as Na⁺ to enter.
2. Inhibitory ion-channel synapse	Opens ion channels that inhibit firing by hyperpolarizing the membrane or shunting excitatory currents.	GABA activates GABA\(_A\) receptors, which conduct Cl⁻. Glycine is another common inhibitory neurotransmitter.
3. Non-channel (second-messenger) synapse	Activates receptors linked to intracellular signaling pathways, indirectly changing ion-channel activity, metabolism, or cellular responses. These effects are often slower and longer-lasting.	Dopamine activates G-protein-coupled receptors that alter intracellular signaling.
4. Neuromuscular junction	A specialized chemical synapse between a motor neuron and a skeletal muscle fiber. It normally triggers a muscle action potential and contraction.	Acetylcholine (ACh) binds nicotinic ACh receptors on the muscle membrane.
5. Electrical synapse	Connects cells through gap junctions, allowing ions and electrical current to pass directly between them. Transmission is very fast and can be bidirectional.	Gap junction channels formed by connexin proteins in vertebrates. No neurotransmitter is required.
Key differences to remember
- Excitatory: generally makes the postsynaptic neuron more likely to fire.
- Inhibitory: generally makes it less likely to fire.
- Second-messenger: changes cellular activity through intracellular signaling pathways.
- Neuromuscular: communicates from a motor neuron to a skeletal muscle fiber.
- Electrical: passes current directly between connected cells rather than relying on neurotransmitter diffusion.
Connection to your two-neuron model: Your Hodgkin–Huxley neurons connected by \(I_{\text{syn}}=g_{\text{syn}}s(V-E_{\text{syn}})\) represent a chemical ion-channel synapse. You can model excitation by choosing an appropriate excitatory reversal potential, or inhibition by using a more negative reversal potential, such as one near the chloride equilibrium potential
*Your Answer Here — which of these five types are E1 and E2 below actually simulating?*
E1 is inhibatory and e2 is excitory.

**Why this matters:** everything in this module (E1 onward) models the fast, ion-channel-based synapses — the first two types. The others (second-messenger, electrical) work on different timescales and mechanisms entirely, which is why we don't simulate them the same way.

### 🎯 Design Problem — Apply the Six-Step Mechanism
**Scenario:** A drug blocks voltage-gated calcium channels at the presynaptic terminal, with no other effect.

**Predict what happens to neurotransmitter release and postsynaptic signaling.**

A) Release drops sharply (or stops), since calcium influx is required to trigger vesicle fusion — postsynaptic signaling fails even though the postsynaptic cell itself is unaffected
B) Release is unaffected, since calcium only matters for the postsynaptic side of the synapse
C) Release increases, since blocking calcium channels removes an inhibitory signal
D) Nothing changes, since neurotransmitter release doesn't require calcium at all

*Reason it out before revealing the answer.*

**Answer: A).**

Calcium influx through voltage-gated channels is the trigger for vesicle fusion and neurotransmitter release — remove it, and the chain breaks at that exact step, even though the postsynaptic receptors themselves are completely normal. This is a real pharmacological principle (this is roughly how some toxins and local anesthetics work).

Notice you didn't need new information to answer this — just applying the six-step chain to a single missing link.

## E1 — Excitatory Logic: AMPA-like Synapse
**Goal:** Observe how a presynaptic spike (Neuron A) triggers a postsynaptic potential (Neuron B) via an excitatory chemical synapse.

### The Biophysical Mechanism
We are simulating an **AMPA-type glutamatergic synapse**. In the real brain, glutamate binds to AMPA receptors, opening a non-selective cation channel with a reversal potential ($E_{rev}$) of approximately **0mV**. Because 0mV is much more positive than the resting potential (-65mV), the resulting current flow is **inward**, causing the cell to depolarize.

**Activity — Ask AI:** *"Why does an AMPA receptor's reversal potential of about 0 mV make it excitatory, while a GABA-A receptor's reversal potential near -65 to -70 mV makes it inhibitory? Explain in terms of driving force."* Paste the response.

*Your Answer Here:*

The key is the driving force, which is the difference between the neuron's membrane potential (\(V_m\)) and the receptor's reversal potential (\(E_{\text{rev}}\)).
\[
\text{Driving force}=V_m-E_{\text{rev}}
\]

- AMPA receptors (excitatory): Their reversal potential is about \(0\text{ mV}\). If the neuron rests at \(-70\text{ mV}\), opening AMPA channels allows a net inward positive-ion current, mainly from Na⁺ entering. This drives the membrane potential toward \(0\text{ mV}\), depolarizing the neuron and making an action potential more likely.
- GABA-A receptors (usually inhibitory): Their reversal potential is often near \(-65\) to \(-70\text{ mV}\). If the membrane potential is \(-50\text{ mV}\), opening GABA-A channels drives the voltage toward that more negative reversal potential, usually hyperpolarizing the neuron. Near resting potential, GABA-A channels can also inhibit firing by shunting excitatory current.
In short: AMPA channels usually pull the membrane voltage toward \(0\text{ mV}\), making firing more likely. GABA-A channels usually pull it toward a more negative potential or reduce the effect of excitation, making firing less likely. The effect depends on the membrane potential relative to the receptor’s reversal potential.

### Student Exercises
1. **The Threshold Quest:** Use the sliders to find the exact `weight` where Neuron B transitions from a sub-threshold "bump" (EPSP) to firing its own Action Potential.
2. **Kinetic Influence:** Set the weight to 0.002. Move `tau` from 2.0 to 8.0. Describe how the **Area** of the signal changes even though the **Peak** remains relatively stable.
3. **Latency Check:** Using the zoomed plot, measure the time between the peak of the Blue spike and the start of the Red rise. Does this match your `delay` slider setting?

In [ ]:
#@title E1 — Interactive Excitatory Logic { display-mode: "form" }
#@markdown Move sliders, then **click the Play button** to update the simulation.

weight = 0.001 #@param {type:"slider", min:0, max:0.01, step:0.0005}
tau = 2.0 #@param {type:"slider", min:0.5, max:10, step:0.5}
delay = 1.0 #@param {type:"slider", min:0.1, max:5, step:0.1}

# 1. Reset & Rebuild
h("forall delete_section()")
somaA = h.Section(name='somaA'); somaB = h.Section(name='somaB')
for s in [somaA, somaB]:
    s.L = s.diam = 20; s.insert('hh')

# 2. Setup Synapse (Excitatory)
syn = h.ExpSyn(somaB(0.5))
syn.tau = tau; syn.e = 0.0
nc = h.NetCon(somaA(0.5)._ref_v, syn, sec=somaA)
nc.threshold = -20; nc.delay = delay; nc.weight[0] = weight

# 3. Stimulus
iclamp = h.IClamp(somaA(0.5))
iclamp.delay = 5; iclamp.dur = 1; iclamp.amp = 2.0

# 4. Run Simulation
vA = mk_rec(somaA); vB = mk_rec(somaB)
h.finitialize(-65); h.continuerun(30)

# 5. Extract Metrics for Title
m = psp_metrics(t, vB)

# 6. Plotting (Single Figure, Two Panes)
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), gridspec_kw={'height_ratios': [1.2, 1]})

# Top Plot: Global Integration
ax1.plot(t, vA, label='Pre (A)', color='C0', alpha=0.5)
ax1.plot(t, vB, label='Post (B)', color='C1', linewidth=2)
ax1.axhline(-65, color='gray', linestyle='--', alpha=0.3)
ax1.set_title(f"Integration Analysis | Peak: {m['Peak']:.2f}mV | Area: {m['Area']:.2f}")
ax1.set_ylabel("Voltage (mV)")
ax1.legend(loc='upper right')

# Bottom Plot: Zoomed Timing
ax2.plot(t, vA, color='C0', alpha=0.5)
ax2.plot(t, vB, color='C1', linewidth=2)
ax2.axhline(-65, color='gray', linestyle='--', alpha=0.3)
ax2.set_xlim(4, 15)
ax2.set_title("Synaptic Latency & Sequence (Zoomed)")
ax2.set_xlabel("Time (ms)"); ax2.set_ylabel("Voltage (mV)")

plt.tight_layout(); plt.show()

---
### Data Presentation & Reproducibility Check
* **Standardization:** Ensure your plots show the **Blue (Pre)** and **Red (Post)** traces clearly.
* **Baseline:** Why must we always indicate the -65mV resting potential when showing sub-threshold EPSPs?
* **Verification:** If you set `weight` to 0, does the Red trace stay perfectly flat? Why is this an important "negative control" for your model?

## E2 — Inhibitory Logic: GABA$_A$-like Synapse
**Goal:** Observe how a presynaptic spike triggers a hyperpolarizing **Inhibitory Post-Synaptic Potential (IPSP)**.

### The Biophysical Mechanism
We are now simulating a **GABA$_A$-type synapse**. In the central nervous system, GABA binds to receptors that open channels primarily permeable to **Chloride ($Cl^-$)**.
* **Reversal Potential ($E_{rev}$):** Typically around **-75mV**.
* **The Driving Force:** Since -75mV is more negative than the resting potential (-65mV), the opening of these channels pulls the membrane potential *downward* (hyperpolarization).

**Activity — Ask AI:** *"What happens to chloride ions when a GABA-A receptor opens, and why does this typically hyperpolarize or stabilize the membrane rather than depolarize it?"* Paste the response.
When a GABA-A receptor opens, it allows chloride ions (\(\mathrm{Cl^-}\)) to cross the neuronal membrane because it is a ligand-gated ion channel activated by the neurotransmitter GABA.
1. What happens to chloride ions?
Chloride movement depends on the membrane potential relative to the chloride equilibrium potential (\(E_{\mathrm{Cl}}\)).
- If the membrane is more positive than \(E_{\mathrm{Cl}}\): chloride ions tend to enter the neuron, bringing negative charge inside and making the membrane potential more negative.
- If the membrane is near \(E_{\mathrm{Cl}}\): there may be little net chloride movement, so the voltage changes very little.
2. Why is this usually inhibitory?
Suppose the membrane potential is \(-50\text{ mV}\) and the chloride equilibrium potential is \(-70\text{ mV}\). Opening GABA-A receptors drives the membrane potential toward \(-70\text{ mV}\), making it more negative. This is hyperpolarization, which generally makes reaching the action-potential threshold harder.
GABA-A receptors can also cause shunting inhibition. Opening them increases membrane conductance, allowing excitatory currents to leak across the membrane and reducing their ability to depolarize the neuron—even if the membrane voltage barely changes.
3. Why doesn't it always hyperpolarize?
The effect depends on the chloride concentration inside and outside the neuron. In some developing neurons, intracellular chloride is relatively high, so the chloride equilibrium potential can be more positive. GABA-A activation may then depolarize the membrane rather than hyperpolarize it.
Bottom line: GABA-A receptors are typically inhibitory because chloride movement drives the membrane toward a relatively negative equilibrium potential, and increased conductance can stabilize the voltage against excitation. Depolarization alone does not necessarily mean excitation; the effect on action-potential firing is what matters.


*Your Answer Here:*
since it is excitory, it wants to make it easier for the neuron to be triggered, and hyperpolarizing lowers the threshold needed.

### Student Exercises
1. **The Polarity Shift:** Run the simulation with the default sliders. Why does the Red trace (Post-B) now go down instead of up?
2. **Kinetics of Inhibition:** Notice the default `tau_inh` is set to **8.0ms**. Compare this to the 2.0ms used in E1. How does this "slower" decay change the integration window for the neuron?
3. **The Shunt:** If you were to change the Resting Potential to -75mV (matching the reversal potential), what would happen to the voltage trace? Would the synapse still be "inhibitory"?

In [ ]:
#@title E2 — Interactive Inhibitory Logic { display-mode: "form" }
#@markdown Move sliders to observe **Inhibitory** signaling. Note the change in $E_{rev}$ and $\tau$.

weight_inh = 0.006 #@param {type:"slider", min:0, max:0.05, step:0.001}
tau_inh = 8.0 #@param {type:"slider", min:1, max:20, step:1}
delay_inh = 1.0 #@param {type:"slider", min:0.1, max:5, step:0.1}

# 1. Reset & Rebuild
h("forall delete_section()")
somaA = h.Section(name='somaA'); somaB = h.Section(name='somaB')
for s in [somaA, somaB]:
    s.L = s.diam = 20; s.insert('hh')

# 2. Setup Synapse (Inhibitory: E_rev = -75mV)
syn = h.ExpSyn(somaB(0.5))
syn.tau = tau_inh
syn.e = -75.0

# 3. Connection
nc = h.NetCon(somaA(0.5)._ref_v, syn, sec=somaA)
nc.threshold = -20; nc.delay = delay_inh; nc.weight[0] = weight_inh

# 4. Trigger Spike in A
iclamp = h.IClamp(somaA(0.5))
iclamp.delay = 5; iclamp.dur = 1; iclamp.amp = 2.0

# 5. Run Simulation
vA = mk_rec(somaA); vB = mk_rec(somaB)
h.finitialize(-65); h.continuerun(40) # Slightly longer to see the slow decay

# 6. Extract Metrics
m = psp_metrics(t, vB)

# 7. Plotting (Consistent with E1)
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), gridspec_kw={'height_ratios': [1.2, 1]})

# Top Plot: Global Integration
ax1.plot(t, vA, label='Pre (A)', color='C0', alpha=0.5)
ax1.plot(t, vB, label='Post (B)', color='C1', linewidth=2)
ax1.axhline(-65, color='gray', linestyle='--', alpha=0.3)
ax1.set_title(f"Inhibitory Analysis | IPSP Peak: {m['Peak']:.2f}mV | Area: {m['Area']:.2f}")
ax1.set_ylabel("Voltage (mV)")
ax1.legend(loc='upper right')

# Bottom Plot: Zoomed Timing
ax2.plot(t, vA, color='C0', alpha=0.5)
ax2.plot(t, vB, color='C1', linewidth=2)
ax2.axhline(-65, color='gray', linestyle='--', alpha=0.3)
ax2.set_xlim(4, 25) # Slightly wider window for the slower IPSP
ax2.set_title("Inhibitory Latency & Decay (Zoomed)")
ax2.set_xlabel("Time (ms)"); ax2.set_ylabel("Voltage (mV)")

plt.tight_layout(); plt.show()

---
### Inhibitory Analysis & Reproducibility
* **Polarity Check:** In the plots above, the Blue trace (Pre) goes UP while the Red trace (Post) goes DOWN. Based on the **Synaptic Variable Key**, why does a reversal potential ($E_{rev}$) of -75mV cause this downward deflection?
* **Integration Window:** Compare the **Area** of this IPSP to the EPSP from E1. Given that inhibitory kinetics (GABA) are often slower than excitatory ones (AMPA), how does this affect the duration of time the neuron is "suppressed" after a single input?
* **Reproducibility:** If you share this notebook with a peer, will they see the same -3.80mV peak? Identify the specific line in the **E0 — Starter** code that ensures every student starts from the same "baseline" voltage.

## E3 — Dynamics Contrast: EPSP vs. IPSP
**Goal:** Run a head-to-head comparison of excitatory and inhibitory signals to characterize their differences in temporal dynamics and magnitude.

### Why Compare?
In neural circuits, the "balance" of excitation and inhibition (E/I balance) is not just about voltage peaks; it is about **timing**.
* **Fast AMPA (Excitatory):** Optimized for speed and precision.
* **Slow GABA (Inhibitory):** Optimized for prolonged suppression and broad integration windows.

### Student Exercises
1. **Area Equalization:** Adjust the `weight` of the inhibitory synapse until its **Area** is approximately equal to the excitatory synapse. Even with equal areas, how do the "shapes" (peak and duration) differ?
2. **The Integration Window:** Looking at the results table, which synapse "controls" the neuron for a longer period of time?
3. **Temporal Precision:** If these two inputs arrived at the same time, which one would "win" the race to change the membrane potential first?

In [ ]:
#@title E3 — EPSP vs IPSP: Comparative Analysis { display-mode: "form" }
import pandas as pd
import matplotlib.pyplot as plt

# 1. Permanent Setup (Outside the loop to prevent crashes)
h("forall delete_section()")
s_comp = h.Section(name='s_comp')
s_comp.L = s_comp.diam = 20
s_comp.insert('hh')

syn_comp = h.ExpSyn(s_comp(0.5))
ns_comp = h.NetStim()
ns_comp.number = 1
ns_comp.start = 5

nc_comp = h.NetCon(ns_comp, syn_comp)
nc_comp.delay = 1

# Record once
v_comp = mk_rec(s_comp)

def run_stable_trial(name, e_rev, tau, weight, color):
    # Update existing objects instead of creating new ones
    syn_comp.e = e_rev
    syn_comp.tau = tau
    nc_comp.weight[0] = weight

    h.finitialize(-65)
    h.continuerun(40)

    # Capture a snapshot of the metrics and trace
    m = psp_metrics(t, v_comp)
    return {
        'Condition': name,
        'Peak (mV)': m['Peak'],
        't_peak (ms)': m['t_peak'],
        'Area': m['Area'],
        'v_data': list(v_comp),
        'color': color
    }

# 2. Execution
results = []
results.append(run_stable_trial("Excitatory (E1)", 0.0, 2.0, 0.002, 'C1'))
results.append(run_stable_trial("Inhibitory (E2)", -75.0, 8.0, 0.008, 'C2'))

# 3. Display Table
df = pd.DataFrame(results)[['Condition', 'Peak (mV)', 't_peak (ms)', 'Area']]
print("\n--- Comparative Metrics ---")
display(df.round(3))

# 4. Plot Comparison
plt.figure(figsize=(8, 4))
for r in results:
    plt.plot(t, r['v_data'], label=r['Condition'], color=r['color'], linewidth=2)
plt.axhline(-65, color='gray', linestyle='--', alpha=0.3)
plt.title("E3: Temporal Dynamics Contrast (AMPA vs. GABA)")
plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)")
plt.legend(); plt.grid(True, alpha=0.2)
plt.show()

---
### E3: The E/I Balance Reflection
*To be completed after reviewing the Comparative Metrics table.*

#### 1. Kinetic Disparity
Based on your results, the **t_peak** for the Inhibitory (E2) synapse is significantly later than the Excitatory (E1) synapse. How does this "slow" inhibition provide a window for excitation to trigger a spike before being suppressed?
> **Answer:** #### 2. The Power of Area
Notice the **Area** metric. A synapse can have a small **Peak** but a large **Area** if its decay ($\tau$) is long. In terms of "total charge" moved across the membrane, which synapse in your current table has a greater overall impact on the cell's excitability?
> **Answer:** #### 3. Predict the Sum
If both these synapses fired at the exact same time ($t=5ms$), would the resulting voltage trace be a simple subtraction of the two? Why or why not? (*Hint: Think about the non-linear nature of the Hodgkin-Huxley membrane.*)
> **Answer:** ```

---

### **Next Step: E4 — The Reversal Sweep**
I noticed in the [latest screenshot](https://colab.research.google.com/drive/1WE8TUKGOOdjsKdwvnNIGdpiKv7DGryCs#scrollTo=957b0fa2) that your **E4** section currently has a different coding style (using a list of values in a `for` loop) compared to the interactive slider versions we built for E1 and E2.

**Would you like me to provide the "Interactive Sweep" version of E4 that uses sliders for the range, matching the layout of the previous sections?**

### 🚫 By-Hand Drill — No AI, No Code
Every synaptic current follows the same Ohm's-law-like form you've used since Set 2: $I_{syn} = g_{syn}(V_m - E_{syn})$.

By hand, using $V_m = -65$ mV (resting):
1. An AMPA synapse opens with $g = 0.5$ nS and $E_{syn} = 0$ mV. Compute $I_{syn}$. Is current flowing into or out of the cell?
2. A GABA-A synapse opens with $g = 0.5$ nS and $E_{syn} = -70$ mV. Compute $I_{syn}$. Is current flowing into or out of the cell?
3. Now suppose the cell has already been depolarized by the AMPA synapse to $V_m = -50$ mV, and the SAME GABA-A synapse opens. Recompute $I_{syn}$ for the inhibitory synapse. Did its magnitude grow or shrink compared to part 2?

*Your Answer Here (show all three calculations) — what does part 3 tell you about why inhibition is often described as more effective on an already-excited cell?*

### 📸 Upload Checkpoint 2 — Driving-Force Calculations, From Memory
**Before you write anything: stop, close this notebook (or at minimum, don't scroll back up).** Without looking back, redo all three driving-force calculations from memory — AMPA at rest, GABA-A at rest, and GABA-A after depolarization to $-50$ mV — from scratch.

Only after you've written what you can recall should you check it against your own earlier work in this notebook. Then photograph your hand-written page and upload it below.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file (`from google.colab import files`) and display it inline.

In [ ]:
# Your image upload + display code here:


## E4 — Reversal Sweep: Finding the "Null Point"
**Goal:** Systematically sweep the Synaptic Reversal Potential ($E_{rev}$) to visualize how the "Driving Force" determines the polarity and magnitude of the post-synaptic response.

### The Concept of Shunting
When $E_{rev}$ is equal to the Resting Potential (**-65mV**), the driving force is zero. The synapse is neither excitatory nor inhibitory in terms of voltage—the trace will remain perfectly flat. However, the synapse is still "active" because it has increased the membrane's conductance. This is known as **Shunting Inhibition**, as it "shunts" (bypasses) other inputs.

### Student Exercises
1. **The Zero-Crossing:** Adjust the sliders to find the exact $E_{rev}$ value where the peak amplitude of the PSP becomes zero. How does this value relate to your resting membrane potential?
2. **Polarity Flip:** As you move $E_{rev}$ from -80mV to -40mV, observe the transition. Does the **t_peak** change significantly as the signal flips from a dip to a bump?
3. **The Shunt:** Set $E_{rev}$ to -65mV. Notice there is no voltage change. Why is this synapse still considered "inhibitory" if it doesn't change the voltage?

In [ ]:
#@title E4 — The Reversal Sweep { display-mode: "form" }
#@markdown Sweep $E_{rev}$ to observe the transition from Hyperpolarization to Depolarization.

e_min = -35 #@param {type:"slider", min:-100, max:0, step:5}
e_max = -45 #@param {type:"slider", min:-100, max:0, step:5}
weight = 0.002 #@param {type:"slider", min:0, max:0.01, step:0.001}

# 1. Stable Setup (Reuse objects to prevent crashes)
h("forall delete_section()")
s_sweep = h.Section(name='s_sweep'); s_sweep.L = s_sweep.diam = 20; s_sweep.insert('hh')
syn_sweep = h.ExpSyn(s_sweep(0.5)); syn_sweep.tau = 5.0
ns_sweep = h.NetStim(); ns_sweep.number = 1; ns_sweep.start = 5
nc_sweep = h.NetCon(ns_sweep, syn_sweep); nc_sweep.delay = 1; nc_sweep.weight[0] = weight
v_sweep = mk_rec(s_sweep)

# 2. Execution of Sweep
sweep_vals = np.linspace(e_min, e_max, 9)
plt.close('all')
plt.figure(figsize=(8, 5))

for e_val in sweep_vals:
    syn_sweep.e = e_val
    h.finitialize(-65)
    h.continuerun(35)

    # Map color: Blue (Hyperpolarizing) to Red (Depolarizing)
    color = plt.cm.coolwarm((e_val - e_min) / (e_max - e_min))
    plt.plot(t, v_sweep, color=color, label=f"Erev: {e_val:.0f}mV", linewidth=1.5)

# 3. Formatting
plt.axhline(-65, color='black', linestyle='--', alpha=0.5, label="Rest (-65mV)")
plt.title(f"E4: Systematic Reversal Sweep (w={weight})")
plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.2)
plt.show()

---
### E4: Reversal Potential Review
* **Zero-Crossing:** Confirm that the "flat" trace occurs at -65mV. This is your biophysical "null point."
* **Driving Force:** Recall the equation $I_{syn} = g_{syn}(V_m - E_{rev})$. When $V_m$ is more positive than $E_{rev}$, the current is outward (inhibitory). When $V_m$ is more negative than $E_{rev}$, the current is inward (excitatory).
* **Next Prediction:** In **E5**, we will fire *two* spikes in quick succession. If the first spike doesn't reach threshold, can the second one "stand on its shoulders" to trigger an action potential?

## E5 — Temporal Summation: The "Shoulder" Effect
**Goal:** Observe how the interval between two presynaptic spikes (Inter-Stimulus Interval or **ISI**) determines the final peak of the postsynaptic response.

### The Biophysical Window
When a second spike arrives before the first EPSP has returned to baseline, the second EPSP "stands on the shoulders" of the first. This is **Temporal Summation**. The efficiency of this process depends on:
* **The Membrane Time Constant ($\tau_m$):** How long the cell "remembers" the voltage change.
* **The Synaptic Time Constant ($\tau_{syn}$):** How long the conductance remains open.

### Student Exercises
1. **The Integration Limit:** Set the ISI to 20ms. Do the two peaks interact at all? Find the "Critical ISI" where the two peaks begin to merge into one single, larger wave.
2. **Saturation:** Set the `weight` high enough that a single spike nearly reaches 0mV. Notice that the second spike doesn't add as much voltage as the first. This is **Sublinear Summation** caused by a reduction in driving force.
3. **Kinetics:** Increase `tau` to 10ms. How does this change the "Temporal Window" in which the two spikes can summate?

In [ ]:
#@title E5 — Temporal Summation Logic { display-mode: "form" }

isi = 14 #@param {type:"slider", min:1, max:30, step:1}
tau = 5 #@param {type:"slider", min:0.5, max:10, step:0.5}
weight = 0.0055 #@param {type:"slider", min:0, max:0.01, step:0.0005}

# 1. Clear memory and rebuild
h("forall delete_section()")
s_sum = h.Section(name='s_sum')
s_sum.L = s_sum.diam = 20
s_sum.insert('hh')

syn_sum = h.ExpSyn(s_sum(0.5))
syn_sum.e = 0
syn_sum.tau = tau

ns_sum = h.NetStim()
ns_sum.number = 2
ns_sum.start = 5
ns_sum.interval = isi

nc_sum = h.NetCon(ns_sum, syn_sum)
nc_sum.delay = 1
nc_sum.weight[0] = weight

# Record
v_sum = mk_rec(s_sum)

# 2. Run
h.finitialize(-65)
h.continuerun(15 + isi + 20)

# 3. Metrics & Plotting
m = psp_metrics(t, v_sum)
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), gridspec_kw={'height_ratios': [1.2, 1]})

# Convert to numpy to avoid TypeError in slicing
t_np = np.array(t)
v_np = np.array(v_sum)

# Top: Global View
ax1.plot(t_np, v_np, color='C1', linewidth=2, label='Post-synaptic (B)')
ax1.axhline(-65, color='gray', linestyle='--', alpha=0.3)
ax1.set_title(f"Temporal Summation (ISI={isi}ms) | Peak: {m['Peak']:.2f}mV")
ax1.set_ylabel("Voltage (mV)")

# Bottom: Derivative (dV/dt) - Now using numpy arrays for slicing
dv = np.diff(v_np) / np.diff(t_np)
ax2.plot(t_np[:-1], dv, color='gray', alpha=0.5)
ax2.set_title("Rate of Change (dV/dt) showing individual arrivals")
ax2.set_xlabel("Time (ms)"); ax2.set_ylabel("V/ms")
ax2.set_xlim(4, 15 + isi)

plt.tight_layout(); plt.show()

---
### E5: Temporal Summation Review
* **The "Shoulder" Effect:** When the ISI is short (e.g., < 10ms), the second EPSP begins before the first has finished. This stacking effect is the primary way a single synapse can drive a neuron toward threshold.
* **Non-linearity:** As the membrane potential gets closer to the reversal potential (0mV), each subsequent spike contributes *less* voltage. This is because the **Driving Force** ($V_m - E_{rev}$) is decreasing.
* **dV/dt Analysis:** Even when the voltage traces look merged, the derivative plot reveals the exact moment each "packet" of neurotransmitter arrived.

## E6 — Spatial Summation & Dendritic Filtering
**Goal:** Quantify how the electrotonic distance of a synapse affects somatic integration.

### The Passive Cable Effect
Dendrites are not perfect wires; they have internal resistance ($R_a$) and "leaky" membranes ($g_{pas}$). As a signal travels from a distant (distal) dendrite toward the soma, it undergoes **Dendritic Filtering**.

### Predict → Verify
Before running the code, record your predictions in the "Student Answer" sections:
1. **Attenuation:** If the distal synapse is at the far end of the dendrite, will the somatic peak be $>50\%$ or $<50\%$ of the proximal peak?
   * *Student Answer:* 2. **Temporal Smearing:** Will the time-to-peak ($t_{peak}$) for the distal input be *earlier* or *later* than the proximal input?
   * *Student Answer:* 3. **Summation:** Will the combined peak (Both) be exactly equal to the sum of the individual peaks (Linear) or smaller (Sublinear)?
   * *Student Answer:* ### 📑 What to Change:
* **`distal_loc`**: Moves the green synapse along the dendrite (0.1 = proximal, 0.9 = distal).
* **`dend_diam`**: Changes the length constant ($\lambda$) by altering internal resistance.
* **`weight`**: Adjusts the synaptic strength for both inputs.

### Student Exercises
1. **The Distance Tax:** Run the simulation for the Proximal input only, then the Distal input only. By what percentage does the somatic peak amplitude drop at your current `distal_loc`?
2. **Speed vs. Distance:** Observe the **t_peak**. Why does the distal signal arrive at the soma later and appear "smeared" (wider) compared to the proximal one?
3. **Diameter Impact:** Set `distal_loc` to 0.9. Now change `dend_diam` from 1.0 to 4.0. How does the "thickness" of the dendrite affect the signal's ability to reach the soma?

---
### E6: Model Validation & Sanity Checks
* **Attenuation Check:** Does the peak amplitude decrease monotonically as `distal_loc` increases? If not, verify that `dendB` is correctly connected to `somaB`.
* **Summation Check:** Calculate `(Peak_Both - Baseline)`. Is it exactly equal to `(Peak_Prox + Peak_Distal - 2*Baseline)`?
    * If it is less, you are observing **Sublinear Summation**.
* **Physics Check:** Does increasing `dend_diam` reduce the attenuation? (Recall that a wider dendrite has lower internal resistance $R_a$).

In [ ]:
#@title E6 — Spatial Summation Logic { display-mode: "form" }

distal_loc = 0.8 #@param {type:"slider", min:0.1, max:0.95, step:0.05}
dend_diam = 1.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}
weight = 0.005 #@param {type:"slider", min:0, max:0.02, step:0.001}

# 1. Persistent Setup (Wipe old sections to keep memory clean)
h("forall delete_section()")
somaB = h.Section(name='somaB'); somaB.L = somaB.diam = 20; somaB.insert('hh')
dendB = h.Section(name='dendB'); dendB.L = 500; dendB.diam = dend_diam; dendB.insert('pas')
dendB.connect(somaB(1))

# 2. Rebuild Synapses
synP = h.ExpSyn(dendB(0.1)) # Proximal
synD = h.ExpSyn(dendB(distal_loc)) # Distal
for s in [synP, synD]: s.e = 0; s.tau = 2.0

# Trigger both at t=5ms
ns = h.NetStim(); ns.number = 1; ns.start = 5
ncP = h.NetCon(ns, synP); ncP.weight[0] = weight; ncP.delay = 1
ncD = h.NetCon(ns, synD); ncD.weight[0] = weight; ncD.delay = 1

# 3. Record from Soma
v_soma = mk_rec(somaB)

# 4. Multi-trial Execution
def run_spatial(wp, wd):
    ncP.weight[0] = wp; ncD.weight[0] = wd
    h.finitialize(-65); h.continuerun(45)
    return np.array(v_soma)

vp_data = run_spatial(weight, 0)
vd_data = run_spatial(0, weight)
vboth_data = run_spatial(weight, weight)
t_np = np.array(t)

# 5. Plotting
plt.close('all')
plt.figure(figsize=(8, 5))
plt.plot(t_np, vp_data, label='Proximal (0.1)', color='C0', linewidth=2)
plt.plot(t_np, vd_data, label=f'Distal ({distal_loc})', color='C2', linewidth=2)
plt.plot(t_np, vboth_data, label='Spatial Summation (Both)', color='black', linestyle='--', alpha=0.7)
plt.axhline(-65, color='gray', linestyle='--', alpha=0.3)
plt.title(f"E6: Spatial Attenuation & Filtering (Dendrite Diam: {dend_diam}µm)")
plt.xlabel("Time (ms)"); plt.ylabel("Soma Voltage (mV)")
plt.legend(); plt.grid(True, alpha=0.2)
plt.show()

### Student Exercises
1. **The Distance Tax:** Run the simulation for the Proximal input only, then the Distal input only. By what percentage does the somatic peak amplitude drop at your current `distal_loc`?
2. **Speed vs. Distance:** Observe the **t_peak**. Why does the distal signal arrive at the soma later and appear "smeared" (wider) compared to the proximal one?
3. **Diameter Impact:** Set `distal_loc` to 0.9. Now change `dend_diam` from 1.0 to 4.0. How does the "thickness" of the dendrite affect the signal's ability to reach the soma?

---
### E6: Model Validation & Sanity Checks
* **Attenuation Check:** Does the peak amplitude decrease monotonically as `distal_loc` increases? If not, verify that `dendB` is correctly connected to `somaB`.
* **Summation Check:** Calculate `(Peak_Both - Baseline)`. Is it exactly equal to `(Peak_Prox + Peak_Distal - 2*Baseline)`?
    * If it is less, you are observing **Sublinear Summation**.
* **Physics Check:** Does increasing `dend_diam` reduce the attenuation? (Recall that a wider dendrite has lower internal resistance $R_a$).

## E7 — Feedforward Inhibition (E→I)
**Goal:** Observe how a fast EPSP followed by a delayed IPSP acts to "clamp" the voltage peak and significantly narrow the integration window for the postsynaptic neuron. This circuit motif allows for precise timing control without necessarily reducing the initial excitatory amplitude.

### The Circuit Motif
In this motif, an extrinsic input excites both a principal neuron and an inhibitory interneuron. This creates a "race" where the EPSP starts first, but is quickly truncated by a delayed IPSP.

### Predict → Verify
* **Window Narrowing:** A smaller $\Delta t$ (inhibition closer to excitation) results in a reduced peak and an earlier return to baseline.
* **Excitability:** Spike probability decreases significantly even if the EPSP onset remains unchanged, as the "window of opportunity" to reach threshold is shortened.

### What to Change:
* **Synaptic Weights:** Adjust `ncE.weight[0]` (excitatory) and `ncI.weight[0]` (inhibitory).
* **Timing:** Change the start time of the inhibitory stimulus relative to the excitatory one ($\Delta t = nsI.start - nsE.start$).
* **Inhibitory Kinetics:** Modify the inhibitory time constant (`synI.tau`).

### Sanity Checks
* **Metric Sweep:** For several $\Delta t$ values, compute the peak voltage, latency, and total area of the postsynaptic response.
* **Clamping Observation:** Show that as the inhibitory input approaches the excitatory input in time (smaller $\Delta t$), the voltage peak is monotonically clamped at lower levels.

### Student Exercises
#### 1. Clamping Sensitivity
**Plot the Peak Voltage vs. $\Delta t$.**
* Identify and mark the specific $\Delta t$ at which the excitatory peak is reduced by **50%** compared to a purely excitatory control.

#### 2. Kinetic Influence on Integration
**Keep $\Delta t$ fixed and vary the inhibitory time constant ($\tau$).**
* **Explain:** Describe the differences in how "tightly" the voltage is clamped and how the decay phase of the PSP changes as inhibition becomes slower or faster.

In [ ]:
#@title E7 — Feedforward Inhibition Logic { display-mode: "form" }

delta_t = 6.5 #@param {type:"slider", min:1, max:15, step:0.5}
weight_exc = 0.017 #@param {type:"slider", min:0, max:0.02, step:0.001}
weight_inh = 0.019 #@param {type:"slider", min:0, max:0.03, step:0.001}
tau_inh = 8.0 #@param {type:"slider", min:1, max:20, step:1}

# 1. Stable Setup
h("forall delete_section()")
somaB = h.Section(name='somaB'); somaB.L = somaB.diam = 20; somaB.insert('hh')

# 2. Synapses
synE = h.ExpSyn(somaB(0.5)); synE.e = 0; synE.tau = 2.0
synI = h.ExpSyn(somaB(0.5)); synI.e = -75; synI.tau = tau_inh

# 3. Stimulators
nsE = h.NetStim(); nsE.number = 1; nsE.start = 5
nsI = h.NetStim(); nsI.number = 1; nsI.start = 5 + delta_t

ncE = h.NetCon(nsE, synE); ncE.weight[0] = weight_exc; ncE.delay = 1
ncI = h.NetCon(nsI, synI); ncI.weight[0] = weight_inh; ncI.delay = 1

v_ff = mk_rec(somaB)

# 4. Multi-trial Execution
def run_ff(we, wi):
    ncE.weight[0] = we; ncI.weight[0] = wi
    h.finitialize(-65); h.continuerun(45)
    return np.array(v_ff)

ve_only = run_ff(weight_exc, 0)
v_combo = run_ff(weight_exc, weight_inh)
t_np = np.array(t)

# 5. Plotting (Fixed Syntax)
plt.close('all')
plt.figure(figsize=(8, 5))
plt.plot(t_np, ve_only, 'C0--', label='Excitation Only (Control)', alpha=0.5)
plt.plot(t_np, v_combo, 'C3', label='FF-Inhibition (E+I)', linewidth=2)
plt.axhline(-65, color='gray', linestyle='--', alpha=0.3)

# Combined f-string and raw string for the Delta symbol
plt.title(fr"E7: Feedforward Clamp ($\Delta t$ = {delta_t}ms)")

plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)")
plt.legend(); plt.grid(True, alpha=0.2)
plt.show()

### E7: Sanity Checks
* **Metric Sweep:** For several $\Delta t$ values, compute the peak voltage, latency, and total area of the postsynaptic response.
* **Clamping Observation:** Show that as the inhibitory input approaches the excitatory input in time (smaller $\Delta t$), the voltage peak is monotonically clamped at lower levels.

### Student Exercises
#### 1. Clamping Sensitivity
**Plot the Peak Voltage vs. $\Delta t$.**
* Identify and mark the specific $\Delta t$ at which the excitatory peak is reduced by **50%** compared to a purely excitatory control.

#### 2. Kinetic Influence on Integration
**Keep $\Delta t$ fixed and vary the inhibitory time constant ($\tau$).**
* **Explain:** Describe the differences in how "tightly" the voltage is clamped and how the decay phase of the PSP changes as inhibition becomes slower or faster.

## E8 — Analysis: Latency, Peak, and Area
**Idea:** Provide a minimal, reusable metrics function to automatically extract key signal features such as the baseline voltage ($V_0$), peak amplitude ($V_{peak}$), time-to-peak ($t_{peak}$), and the total area under the curve (integral).

### Predict → Verify
1. **Area vs. Peak:** If inhibition narrows the EPSP but doesn't change the initial slope, will the *Peak* or the *Area* show a larger percentage decrease?
   * *Student Answer:*
2. **Latency:** Does adding inhibition shift the *measured* $t_{peak}$ of the EPSP earlier or later?
   * *Student Answer:*

### What to Change:
* **Baseline Window:** Adjust the time range used to calculate the reference $V_0$ in the `psp_metrics` function.
* **Detection Thresholds:** Modify the criteria used to identify the start and end of a synaptic event for area calculations.

### Model Validation
* **Validation Run:** Execute the helper function on the most recent simulation trace.
* **Verification:** Confirm that the output units (e.g., mV, ms) are correct and that the signs (positive for EPSPs, negative for IPSPs) match the visual data.

### Exercises
#### 1. Expanding the Toolkit
Update the metrics function to include two additional calculations:
* **Half-width:** The duration of the signal at 50% of its peak amplitude.
* **10-90% Rise Time:** The time it takes for the signal to transition from 10% to 90% of its peak.
* **Compare:** Use these new metrics to quantify the specific differences between a standard EPSP and a standard IPSP from your earlier simulations.

In [ ]:
#@title E8 — Run Automated Metrics { display-mode: "form" }
import pandas as pd

# 1. Extract metrics using the helper from E0
# We analyze the traces generated in E7
m_e = psp_metrics(t_np, ve_only)
m_c = psp_metrics(t_np, v_combo)

# 2. Display as a Table
df = pd.DataFrame({
    "Metric": ["Baseline (V0)", "Peak Amp (mV)", "t_peak (ms)", "Total Area"],
    "Exc Only (Control)": [f"{m_e['V0']:.2f}", f"{m_e['Peak']:.2f}", f"{m_e['t_peak']:.2f}", f"{m_e['Area']:.2f}"],
    "E + I (Clamped)": [f"{m_c['V0']:.2f}", f"{m_c['Peak']:.2f}", f"{m_c['t_peak']:.2f}", f"{m_c['Area']:.2f}"]
})

print("--- Automated Synaptic Analysis ---")
display(df)

# 3. Visual Validation Plot
plt.figure(figsize=(8, 3))
plt.plot(t_np, v_combo, color='C3', label='Post-synaptic (E+I)')
plt.fill_between(t_np, -65, v_combo, color='C3', alpha=0.1, label='Integrated Area')
plt.axvline(m_c['t_peak'], color='black', linestyle=':', label='Peak Detected')
plt.title("E8: Metric Extraction Visualization")
plt.legend(); plt.xlabel("Time (ms)"); plt.ylabel("mV"); plt.show()

### E8: Model Validation
* **Validation Run:** Execute the helper function on the most recent simulation trace to verify the alignment of detection markers.
* **Verification:** Confirm that the output units (e.g., mV, ms) are correct and that the signs (positive for EPSPs, negative for IPSPs) match the visual data.
* **Peak Alignment:** Ensure the vertical "Peak Detected" line aligns perfectly with the highest point of the voltage trace.

### Exercises
#### 1. Expanding the Toolkit
Update the `psp_metrics` function (or create a new analysis block) to include two additional calculations:
* **Half-width:** The duration of the signal (in ms) at 50% of its peak amplitude.
* **10-90% Rise Time:** The time it takes for the signal to transition from 10% to 90% of its peak.
* **Compare:** Use these new metrics to quantify the specific differences between a standard EPSP and a standard IPSP from your earlier simulations.

#### 2. Percentage Analysis
* **Inhibitory Impact:** Use the automated table to calculate the percentage reduction in **Total Area** caused by the feedforward inhibition in E7.
* **Discussion:** Is the reduction in **Area** more or less significant than the reduction in **Peak Amplitude**? Why might the area be a better measure of "total synaptic drive"?

## E9 — Synaptic Playground
**Idea:** Free exploration. Systematically vary synaptic parameters to practice **Predict → Verify** reasoning. This section is designed to test your intuition about how electrochemical gradients and channel kinetics shape the post-synaptic response.

### What to Change:
* **Synaptic Identity:** Reversal potential ($E_{rev}$).
* **Temporal Profile:** Decay time constant ($\tau$).
* **Strength:** Synaptic weight ($w$).
* **Timing:** Axonal or stimulus delay.
* **Documentation:** Record the specific parameter tuple used for every run.

### Model Validation
For every experimental configuration, consistently record and verify the following metrics:
* **Baseline voltage ($V_0$)**
* **Peak amplitude ($V_{peak}$)**
* **Latency to peak ($t_{peak}$)**
* **Total area (integral)**

### Exercises
#### 1. Predict → Verify Log
Perform three distinct "custom" runs. For each, write down your prediction of the change (e.g., "Increasing $\tau$ while decreasing weight will result in a wider but shorter PSP") before executing the code.

* **Run 1:** [Parameters] -> [Prediction] -> [Result]
* **Run 2:** [Parameters] -> [Prediction] -> [Result]
* **Run 3:** [Parameters] -> [Prediction] -> [Result]

#### 2. The "Ultimate Clamp" Challenge
Find a combination of $E_{rev}$, $Weight$, and $Tau$ that allows you to maintain the membrane potential within **1mV** of the resting potential, even when a massive excitatory weight (0.05) is applied. What does this tell you about the power of shunting inhibition?

**Make It Your Own (optional AI-assisted extension).** Extend the playground:
- Add a **third synapse type** (e.g., a slow NMDA-like conductance with a much longer decay $\tau$) alongside the existing AMPA-like and GABA$_A$-like synapses, and compare its summation behavior to the other two.
- Add an automated **E/I balance point finder**: sweep excitatory/inhibitory weight ratios and report the ratio at which net depolarization crosses zero.

> **Suggested prompt:** *"Here is my NEURON synaptic playground code: [paste it]. I want to add [your extension] without changing how the existing synapses work. Show me what to add and where."*

*Your Answer Here (what you added, and what it showed you):*

In [ ]:
#@title E9 — Synaptic Playground Master Controller { display-mode: "form" }

e_rev = -71 #@param {type:"slider", min:-90, max:20, step:1}
tau = 10 #@param {type:"slider", min:1, max:50, step:1}
weight = 0.005 #@param {type:"slider", min:0, max:0.05, step:0.001}
delay = 1 #@param {type:"slider", min:1, max:20, step:0.5}

# 1. Setup Environment
h("forall delete_section()")
soma_p = h.Section(name='soma_p'); soma_p.L = soma_p.diam = 20; soma_p.insert('hh')

# 2. Universal Synapse
syn = h.ExpSyn(soma_p(0.5))
syn.e = e_rev
syn.tau = tau

ns = h.NetStim(); ns.number = 1; ns.start = 5
nc = h.NetCon(ns, syn); nc.weight[0] = weight; nc.delay = delay

v_rec = mk_rec(soma_p)

# 3. Execution
h.finitialize(-65)
h.continuerun(60)
t_np = np.array(t); v_np = np.array(v_rec)

# 4. Automated Analysis
m = psp_metrics(t_np, v_np)

# 5. Plotting & Results
plt.close('all')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [2, 1]})

ax1.plot(t_np, v_np, color='black', linewidth=2)
ax1.axhline(-65, color='gray', linestyle='--', alpha=0.5)
ax1.set_title(fr"Playground: $E_{{rev}}$={e_rev}mV, $\tau$={tau}ms, $w$={weight}")
ax1.set_xlabel("Time (ms)"); ax1.set_ylabel("Soma Voltage (mV)")

# Summary Table on Plot
stats_text = (f"Baseline: {m['V0']:.2f}mV\n"
              f"Peak: {m['Peak']:.2f}mV\n"
              f"T-Peak: {m['t_peak']:.2f}ms\n"
              f"Area: {m['Area']:.2f}")
ax2.axis('off')
ax2.text(0.1, 0.5, stats_text, fontsize=12, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.show()

### Reflection
* **Parameter Interaction:** How do $E_{syn}$ and $\tau$ jointly control PSP amplitude and timing?
* **Inhibitory Polarity:** When does inhibitory input look depolarizing yet stay functionally inhibitory?
* **Summation Dynamics:** How do ISI and electrotonic distance interact in postsynaptic integration?

---
## Design Challenges: A Three-Compartment Circuit
So far you've worked with single synapses onto a single compartment. Real neurons integrate many synapses across a branched dendritic tree. Consider a **three-compartment model**: two dendrite compartments (Dendrite 1, Dendrite 2) feeding into a soma, with two input sources — Source A (excitatory) and Source B (inhibitory) — each able to contact either dendrite.

This section is a set of **design challenges**: you're not just observing anymore, you're engineering a specific outcome.

### 📸 Upload Checkpoint 3 — Sketch the Three-Compartment Circuit
Before starting the design challenges below, sketch the three-compartment setup yourself, from the description above — two dendrite compartments, the soma, and where Source A (excitatory) and Source B (inhibitory) connect. Do this from your own understanding of the text, not by copying a diagram.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file and display it inline.

In [ ]:
# Your image upload + display code here:


### Design Challenge 1 — Rate-Coded Temporal Summation
E5 showed you temporal summation with just two pulses. Real inputs arrive as trains.

**Activity — vibe-code it:** extend your E5-style rig (or build a new one) to deliver a *train* of excitatory pulses at a fixed interval, and measure the resulting steady-state firing rate at the soma. Repeat for several intervals, keeping (interval × number of pulses) constant at 50, so total input "dose" stays fixed while rate varies.

> **Suggested prompt:** *"Here is my NEURON excitatory synapse code: [paste it]. I want to deliver a train of N pulses at a fixed interval using NetStim, then measure the postsynaptic firing rate. Help me sweep interval values from 10 down to 0.5 ms while keeping interval times number equal to 50."*

Plot output firing rate vs. input rate — this is the synapse's **input-output transfer function**.

*Your Answer Here — is the relationship close to linear, or does it saturate at high input rates?*

In [ ]:
# Your rate-coded temporal summation sweep code here:

# If needed, run this in a separate Colab cell first:
# %pip install neuron scipy matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
from neuron import h

h.load_file("stdrun.hoc")

# -------------------------------------------------------
# 1. Build the three-compartment neuron
# -------------------------------------------------------
h.dt = 0.025
h.steps_per_ms = 1.0 / h.dt
h.secondorder = 0
h.tstop = 130

dend = h.Section(name="dendrite")
soma = h.Section(name="soma")
ais = h.Section(name="axon_initial_segment")

dend.L, dend.diam = 200, 2
soma.L, soma.diam = 20, 20
ais.L, ais.diam = 30, 1.5

dend.nseg = soma.nseg = ais.nseg = 1
dend.connect(soma(0))
ais.connect(soma(1))

for sec in (dend, soma, ais):
    sec.Ra = 100
    sec.cm = 1
    sec.insert("hh")

# Excitatory synapse: Source A
exc = h.ExpSyn(dend(0.5))
exc.tau = 2.0
exc.e = 0.0

# Inhibitory synapse: Source B
inh = h.Exp2Syn(dend(0.5))
inh.tau1 = 0.5
inh.tau2 = 8.0
inh.e = -75.0

# Source A emits three events
a_times = h.Vector([20.0, 60.0, 100.0])
srcA = h.VecStim()
srcA.play(a_times)
ncA = h.NetCon(srcA, exc)
ncA.delay = 0
ncA.weight[0] = 0.04  # uS; will be calibrated below

# Source B event times will be changed during the sweep
b_times = h.Vector()
srcB = h.VecStim()
srcB.play(b_times)
ncB = h.NetCon(srcB, inh)
ncB.delay = 0
ncB.weight[0] = 0.05  # uS; varied during sweep

# Record axon voltage and time
t_rec = h.Vector().record(h._ref_t)
v_rec = h.Vector().record(ais(0.5)._ref_v)


def run_trial(g_exc, g_inh, delay, n_pulses, interval):
    """Delay is relative to Source A's second event at 60 ms."""
    exc.g = 0  # ExpSyn has no g field; event strength is NetCon weight
    ncA.weight[0] = g_exc
    ncB.weight[0] = g_inh

    # Negative delay means inhibition starts before the second input.
    pulse_times = [
        60.0 + delay + i * interval
        for i in range(n_pulses)
    ]
    pulse_times = [x for x in pulse_times if x >= 0]
    b_times.from_python(pulse_times)

    h.finitialize(-65)
    h.continuerun(h.tstop)

    t = np.asarray(t_rec)
    v = np.asarray(v_rec)

    # Detect upward crossings of 0 mV; enforce 2 ms refractory spacing.
    crossings = np.where((v[:-1] < 0) & (v[1:] >= 0))[0] + 1
    spike_times = []
    for idx in crossings:
        ts = t[idx]
        if not spike_times or ts - spike_times[-1] >= 2.0:
            spike_times.append(ts)

    return t, v, np.asarray(spike_times), pulse_times


def spike_pattern(spikes):
    """Count spikes in broad windows after the three excitatory events."""
    windows = [(10, 40), (45, 75), (85, 115)]
    return tuple(
        int(np.sum((spikes >= lo) & (spikes <= hi)))
        for lo, hi in windows
    )


# -------------------------------------------------------
# 2. Calibrate Source A to produce exactly three spikes
# -------------------------------------------------------
candidate_g_exc = [0.01, 0.02, 0.04, 0.06, 0.08, 0.12]
chosen_g_exc = None

for ge in candidate_g_exc:
    t, v, spikes, _ = run_trial(
        g_exc=ge, g_inh=0.0,
        delay=0, n_pulses=0, interval=5
    )
    if spike_pattern(spikes) == (1, 1, 1):
        chosen_g_exc = ge
        break

if chosen_g_exc is None:
    raise RuntimeError(
        "Source A did not produce one spike per input with the tested "
        "weights. Try expanding candidate_g_exc or adjusting dendrite "
        "geometry and synaptic strength."
    )

print("Excitatory weight selected:", chosen_g_exc, "uS")
print("Uninhibited spike times (ms):", np.round(spikes, 2))


# -------------------------------------------------------
# 3. Search delay, pulse count, interval, and inhibition
# -------------------------------------------------------
delays = range(-8, 13, 2)       # ms relative to second input
pulse_counts = [1, 2, 3]
intervals = [2, 4, 6, 8]       # ms between inhibitory pulses
inh_weights = [0.02, 0.05, 0.10]  # uS

solutions = []

for gi in inh_weights:
    for delay in delays:
        for n in pulse_counts:
            for interval in intervals:
                t, v, spikes, pulse_times = run_trial(
                    g_exc=chosen_g_exc,
                    g_inh=gi,
                    delay=delay,
                    n_pulses=n,
                    interval=interval
                )

                pattern = spike_pattern(spikes)

                # First and third input evoke spikes; second does not.
                if pattern == (1, 0, 1):
                    solutions.append({
                        "g_inh": gi,
                        "delay": delay,
                        "n": n,
                        "interval": interval,
                        "spikes": spikes.copy(),
                        "pulse_times": pulse_times.copy()
                    })

if not solutions:
    print(
        "No successful combination found in this sweep. "
        "Try a wider delay range, stronger inhibition, or a finer "
        "excitatory-weight calibration."
    )
else:
    # Show a few successful combinations
    print(f"\nFound {len(solutions)} successful combinations.")
    for s in solutions[:10]:
        print(
            f"g_inh={s['g_inh']:.3f} uS, "
            f"delay={s['delay']} ms, "
            f"pulses={s['n']}, "
            f"interval={s['interval']} ms, "
            f"spikes={np.round(s['spikes'], 2)} ms"
        )

    # Plot the first successful result
    best = solutions[0]
    t, v, spikes, pulse_times = run_trial(
        g_exc=chosen_g_exc,
        g_inh=best["g_inh"],
        delay=best["delay"],
        n_pulses=best["n"],
        interval=best["interval"]
    )

    plt.figure(figsize=(11, 5))
    plt.plot(t, v, label="Axon initial segment voltage")
    plt.axhline(0, linestyle="--", linewidth=1, label="Spike threshold")
    for x in [20, 60, 100]:
        plt.axvline(x, linestyle=":", alpha=0.5)
    for i, x in enumerate(pulse_times):
        plt.axvline(
            x, color="red", linestyle="--", alpha=0.6,
            label="Inhibitory pulse" if i == 0 else None
        )

    plt.xlabel("Time (ms)")
    plt.ylabel("Membrane potential (mV)")
    plt.title(
        f"Selective suppression: delay={best['delay']} ms, "
        f"{best['n']} pulses, interval={best['interval']} ms"
    )
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Final spike times (ms):", np.round(spikes, 2))
    print("Spike pattern:", spike_pattern(spikes))


### Design Challenge 2 — Surgically Suppress One Spike
Here's a precision design problem. Source A (excitatory) fires a train that produces **three action potentials** at the soma. Your job: using **only** Source B (inhibitory) — its delay, number of pulses, and interval — suppress **just the middle spike**, while the **first and last spikes must survive untouched**.

You may not change anything about Source A.

**Activity — vibe-code it:**

> **Suggested prompt:** *"Here is my three-compartment NEURON model with excitatory Source A producing 3 spikes and inhibitory Source B available on the same dendrite. Help me find inhibitory timing (delay, number of pulses, interval) that eliminates only the second spike, leaving the first and third spikes intact."*

*Your Answer Here — report the exact Source B parameters that worked, and explain in one sentence why timing (not just strength) is what makes this surgical precision possible.*
The timing is what gives the ions time to develop voltage and current, at the necesarry strength to drive a trigger.

In [1]:
# Your precision-inhibition design code here:

# If needed, run this in a separate Colab cell first:
# %pip install neuron scipy matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
from neuron import h

h.load_file("stdrun.hoc")

# -------------------------------------------------------
# 1. Build the three-compartment neuron
# -------------------------------------------------------
h.dt = 0.025
h.steps_per_ms = 1.0 / h.dt
h.secondorder = 0
h.tstop = 130

dend = h.Section(name="dendrite")
soma = h.Section(name="soma")
ais = h.Section(name="axon_initial_segment")

dend.L, dend.diam = 200, 2
soma.L, soma.diam = 20, 20
ais.L, ais.diam = 30, 1.5

dend.nseg = soma.nseg = ais.nseg = 1
dend.connect(soma(0))
ais.connect(soma(1))

for sec in (dend, soma, ais):
    sec.Ra = 100
    sec.cm = 1
    sec.insert("hh")

# Excitatory synapse: Source A
exc = h.ExpSyn(dend(0.5))
exc.tau = 2.0
exc.e = 0.0

# Inhibitory synapse: Source B
inh = h.Exp2Syn(dend(0.5))
inh.tau1 = 0.5
inh.tau2 = 8.0
inh.e = -75.0

# Source A emits three events
a_times = h.Vector([20.0, 60.0, 100.0])
srcA = h.VecStim()
srcA.play(a_times)
ncA = h.NetCon(srcA, exc)
ncA.delay = 0
ncA.weight[0] = 0.04  # uS; will be calibrated below

# Source B event times will be changed during the sweep
b_times = h.Vector()
srcB = h.VecStim()
srcB.play(b_times)
ncB = h.NetCon(srcB, inh)
ncB.delay = 0
ncB.weight[0] = 0.05  # uS; varied during sweep

# Record axon voltage and time
t_rec = h.Vector().record(h._ref_t)
v_rec = h.Vector().record(ais(0.5)._ref_v)


def run_trial(g_exc, g_inh, delay, n_pulses, interval):
    """Delay is relative to Source A's second event at 60 ms."""
    exc.g = 0  # ExpSyn has no g field; event strength is NetCon weight
    ncA.weight[0] = g_exc
    ncB.weight[0] = g_inh

    # Negative delay means inhibition starts before the second input.
    pulse_times = [
        60.0 + delay + i * interval
        for i in range(n_pulses)
    ]
    pulse_times = [x for x in pulse_times if x >= 0]
    b_times.from_python(pulse_times)

    h.finitialize(-65)
    h.continuerun(h.tstop)

    t = np.asarray(t_rec)
    v = np.asarray(v_rec)

    # Detect upward crossings of 0 mV; enforce 2 ms refractory spacing.
    crossings = np.where((v[:-1] < 0) & (v[1:] >= 0))[0] + 1
    spike_times = []
    for idx in crossings:
        ts = t[idx]
        if not spike_times or ts - spike_times[-1] >= 2.0:
            spike_times.append(ts)

    return t, v, np.asarray(spike_times), pulse_times


def spike_pattern(spikes):
    """Count spikes in broad windows after the three excitatory events."""
    windows = [(10, 40), (45, 75), (85, 115)]
    return tuple(
        int(np.sum((spikes >= lo) & (spikes <= hi)))
        for lo, hi in windows
    )


# -------------------------------------------------------
# 2. Calibrate Source A to produce exactly three spikes
# -------------------------------------------------------
candidate_g_exc = [0.01, 0.02, 0.04, 0.06, 0.08, 0.12]
chosen_g_exc = None

for ge in candidate_g_exc:
    t, v, spikes, _ = run_trial(
        g_exc=ge, g_inh=0.0,
        delay=0, n_pulses=0, interval=5
    )
    if spike_pattern(spikes) == (1, 1, 1):
        chosen_g_exc = ge
        break

if chosen_g_exc is None:
    raise RuntimeError(
        "Source A did not produce one spike per input with the tested "
        "weights. Try expanding candidate_g_exc or adjusting dendrite "
        "geometry and synaptic strength."
    )

print("Excitatory weight selected:", chosen_g_exc, "uS")
print("Uninhibited spike times (ms):", np.round(spikes, 2))


# -------------------------------------------------------
# 3. Search delay, pulse count, interval, and inhibition
# -------------------------------------------------------
delays = range(-8, 13, 2)       # ms relative to second input
pulse_counts = [1, 2, 3]
intervals = [2, 4, 6, 8]       # ms between inhibitory pulses
inh_weights = [0.02, 0.05, 0.10]  # uS

solutions = []

for gi in inh_weights:
    for delay in delays:
        for n in pulse_counts:
            for interval in intervals:
                t, v, spikes, pulse_times = run_trial(
                    g_exc=chosen_g_exc,
                    g_inh=gi,
                    delay=delay,
                    n_pulses=n,
                    interval=interval
                )

                pattern = spike_pattern(spikes)

                # First and third input evoke spikes; second does not.
                if pattern == (1, 0, 1):
                    solutions.append({
                        "g_inh": gi,
                        "delay": delay,
                        "n": n,
                        "interval": interval,
                        "spikes": spikes.copy(),
                        "pulse_times": pulse_times.copy()
                    })

if not solutions:
    print(
        "No successful combination found in this sweep. "
        "Try a wider delay range, stronger inhibition, or a finer "
        "excitatory-weight calibration."
    )
else:
    # Show a few successful combinations
    print(f"\nFound {len(solutions)} successful combinations.")
    for s in solutions[:10]:
        print(
            f"g_inh={s['g_inh']:.3f} uS, "
            f"delay={s['delay']} ms, "
            f"pulses={s['n']}, "
            f"interval={s['interval']} ms, "
            f"spikes={np.round(s['spikes'], 2)} ms"
        )

    # Plot the first successful result
    best = solutions[0]
    t, v, spikes, pulse_times = run_trial(
        g_exc=chosen_g_exc,
        g_inh=best["g_inh"],
        delay=best["delay"],
        n_pulses=best["n"],
        interval=best["interval"]
    )

    plt.figure(figsize=(11, 5))
    plt.plot(t, v, label="Axon initial segment voltage")
    plt.axhline(0, linestyle="--", linewidth=1, label="Spike threshold")
    for x in [20, 60, 100]:
        plt.axvline(x, linestyle=":", alpha=0.5)
    for i, x in enumerate(pulse_times):
        plt.axvline(
            x, color="red", linestyle="--", alpha=0.6,
            label="Inhibitory pulse" if i == 0 else None
        )

    plt.xlabel("Time (ms)")
    plt.ylabel("Membrane potential (mV)")
    plt.title(
        f"Selective suppression: delay={best['delay']} ms, "
        f"{best['n']} pulses, interval={best['interval']} ms"
    )
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Final spike times (ms):", np.round(spikes, 2))
    print("Spike pattern:", spike_pattern(spikes))


ModuleNotFoundError: No module named 'neuron'

### Design Challenge 3 — Does Location Matter?
You solved Challenge 2 with inhibition on the same dendrite as the excitation. Now test whether **location** matters as much as **timing**.

**Setup A:** Excitation and inhibition both on Dendrite 1 (same as Challenge 2).
**Setup B:** Excitation on Dendrite 1, inhibition on Dendrite 2 (different compartments, same soma).

For a fair comparison, sweep the inhibitory synaptic weight (to one decimal place) in both setups and find the *minimum* weight needed to suppress the middle spike in each.

*Your Answer Here — which setup needed a smaller inhibitory weight to achieve the same suppression? Propose a mechanistic reason, in terms of how the two currents interact locally vs. after already spreading through the cable.*

**Now reverse it entirely:** put the excitatory input on Dendrite 2 and the inhibitory input on Dendrite 1. Repeat the comparison.

*Your Answer Here — does the effectiveness pattern flip, stay the same, or do something more complicated? What does this tell you about designing real dendritic circuits, where excitatory and inhibitory synapses are never randomly placed?*

In [ ]:
# Your location-comparison sweep code here:

# If needed, run this in a separate Colab cell first:
# %pip install neuron scipy matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
from neuron import h

h.load_file("stdrun.hoc")

# -------------------------------------------------------
# 1. Build the three-compartment neuron
# -------------------------------------------------------
h.dt = 0.025
h.steps_per_ms = 1.0 / h.dt
h.secondorder = 0
h.tstop = 130

dend = h.Section(name="dendrite")
soma = h.Section(name="soma")
ais = h.Section(name="axon_initial_segment")

dend.L, dend.diam = 200, 2
soma.L, soma.diam = 20, 20
ais.L, ais.diam = 30, 1.5

dend.nseg = soma.nseg = ais.nseg = 1
dend.connect(soma(0))
ais.connect(soma(1))

for sec in (dend, soma, ais):
    sec.Ra = 100
    sec.cm = 1
    sec.insert("hh")

# Excitatory synapse: Source A
exc = h.ExpSyn(dend(0.5))
exc.tau = 2.0
exc.e = 0.0

# Inhibitory synapse: Source B
inh = h.Exp2Syn(dend(0.5))
inh.tau1 = 0.5
inh.tau2 = 8.0
inh.e = -75.0

# Source A emits three events
a_times = h.Vector([20.0, 60.0, 100.0])
srcA = h.VecStim()
srcA.play(a_times)
ncA = h.NetCon(srcA, exc)
ncA.delay = 0
ncA.weight[0] = 0.04  # uS; will be calibrated below

# Source B event times will be changed during the sweep
b_times = h.Vector()
srcB = h.VecStim()
srcB.play(b_times)
ncB = h.NetCon(srcB, inh)
ncB.delay = 0
ncB.weight[0] = 0.05  # uS; varied during sweep

# Record axon voltage and time
t_rec = h.Vector().record(h._ref_t)
v_rec = h.Vector().record(ais(0.5)._ref_v)


def run_trial(g_exc, g_inh, delay, n_pulses, interval):
    """Delay is relative to Source A's second event at 60 ms."""
    exc.g = 0  # ExpSyn has no g field; event strength is NetCon weight
    ncA.weight[0] = g_exc
    ncB.weight[0] = g_inh

    # Negative delay means inhibition starts before the second input.
    pulse_times = [
        60.0 + delay + i * interval
        for i in range(n_pulses)
    ]
    pulse_times = [x for x in pulse_times if x >= 0]
    b_times.from_python(pulse_times)

    h.finitialize(-65)
    h.continuerun(h.tstop)

    t = np.asarray(t_rec)
    v = np.asarray(v_rec)

    # Detect upward crossings of 0 mV; enforce 2 ms refractory spacing.
    crossings = np.where((v[:-1] < 0) & (v[1:] >= 0))[0] + 1
    spike_times = []
    for idx in crossings:
        ts = t[idx]
        if not spike_times or ts - spike_times[-1] >= 2.0:
            spike_times.append(ts)

    return t, v, np.asarray(spike_times), pulse_times


def spike_pattern(spikes):
    """Count spikes in broad windows after the three excitatory events."""
    windows = [(10, 40), (45, 75), (85, 115)]
    return tuple(
        int(np.sum((spikes >= lo) & (spikes <= hi)))
        for lo, hi in windows
    )


# -------------------------------------------------------
# 2. Calibrate Source A to produce exactly three spikes
# -------------------------------------------------------
candidate_g_exc = [0.01, 0.02, 0.04, 0.06, 0.08, 0.12]
chosen_g_exc = None

for ge in candidate_g_exc:
    t, v, spikes, _ = run_trial(
        g_exc=ge, g_inh=0.0,
        delay=0, n_pulses=0, interval=5
    )
    if spike_pattern(spikes) == (1, 1, 1):
        chosen_g_exc = ge
        break

if chosen_g_exc is None:
    raise RuntimeError(
        "Source A did not produce one spike per input with the tested "
        "weights. Try expanding candidate_g_exc or adjusting dendrite "
        "geometry and synaptic strength."
    )

print("Excitatory weight selected:", chosen_g_exc, "uS")
print("Uninhibited spike times (ms):", np.round(spikes, 2))


# -------------------------------------------------------
# 3. Search delay, pulse count, interval, and inhibition
# -------------------------------------------------------
delays = range(-8, 13, 2)       # ms relative to second input
pulse_counts = [1, 2, 3]
intervals = [2, 4, 6, 8]       # ms between inhibitory pulses
inh_weights = [0.02, 0.05, 0.10]  # uS

solutions = []

for gi in inh_weights:
    for delay in delays:
        for n in pulse_counts:
            for interval in intervals:
                t, v, spikes, pulse_times = run_trial(
                    g_exc=chosen_g_exc,
                    g_inh=gi,
                    delay=delay,
                    n_pulses=n,
                    interval=interval
                )

                pattern = spike_pattern(spikes)

                # First and third input evoke spikes; second does not.
                if pattern == (1, 0, 1):
                    solutions.append({
                        "g_inh": gi,
                        "delay": delay,
                        "n": n,
                        "interval": interval,
                        "spikes": spikes.copy(),
                        "pulse_times": pulse_times.copy()
                    })

if not solutions:
    print(
        "No successful combination found in this sweep. "
        "Try a wider delay range, stronger inhibition, or a finer "
        "excitatory-weight calibration."
    )
else:
    # Show a few successful combinations
    print(f"\nFound {len(solutions)} successful combinations.")
    for s in solutions[:10]:
        print(
            f"g_inh={s['g_inh']:.3f} uS, "
            f"delay={s['delay']} ms, "
            f"pulses={s['n']}, "
            f"interval={s['interval']} ms, "
            f"spikes={np.round(s['spikes'], 2)} ms"
        )

    # Plot the first successful result
    best = solutions[0]
    t, v, spikes, pulse_times = run_trial(
        g_exc=chosen_g_exc,
        g_inh=best["g_inh"],
        delay=best["delay"],
        n_pulses=best["n"],
        interval=best["interval"]
    )

    plt.figure(figsize=(11, 5))
    plt.plot(t, v, label="Axon initial segment voltage")
    plt.axhline(0, linestyle="--", linewidth=1, label="Spike threshold")
    for x in [20, 60, 100]:
        plt.axvline(x, linestyle=":", alpha=0.5)
    for i, x in enumerate(pulse_times):
        plt.axvline(
            x, color="red", linestyle="--", alpha=0.6,
            label="Inhibitory pulse" if i == 0 else None
        )

    plt.xlabel("Time (ms)")
    plt.ylabel("Membrane potential (mV)")
    plt.title(
        f"Selective suppression: delay={best['delay']} ms, "
        f"{best['n']} pulses, interval={best['interval']} ms"
    )
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Final spike times (ms):", np.round(spikes, 2))
    print("Spike pattern:", spike_pattern(spikes))


---
## Homework: This Is What Quizzes and Exams Will Look Like
- **By hand, no AI:** the AMPA/GABA-A driving-force calculation (three parts)
- **Design deliverables:** the rate-coded transfer function, the precision-suppression parameters, and the location-comparison results, all with your actual numbers reported
- **Quantitative measurements:** reversal potentials (E4), summation windows (E5), spatial attenuation (E6)

---
## Knowledge Bank — Set 6: Synapses and Communication

**This is the outcome of this module.** Quizzes, the midterm, and the final draw directly from these questions — know them well. For anything you're unsure about, use an AI assistant to research it, and add what you find as your own notes right in this notebook.

### A. Synapse Biology and Types
1. List the six steps from an action potential arriving at a presynaptic terminal to a possible new action potential in the postsynaptic cell.
2. Name the five types of synapses and one typical neurotransmitter or mechanism for each.
3. Why can't an action potential simply jump across the synaptic cleft?

### B. Excitatory and Inhibitory Currents
4. Write the driving-force equation for synaptic current, and explain what each term means.
5. Why does an AMPA synapse depolarize the cell while a GABA-A synapse hyperpolarizes or stabilizes it?
6. By hand: why does the same inhibitory synapse produce a larger current when the cell is already depolarized than when it's at rest?
7. Why is an inhibitory synapse with $E_{syn}=-65$ mV "silent" in a resting cell but "powerful" during excitation?

### C. Summation and Timing
8. Distinguish spatial summation from temporal summation.
9. What is "sublinear summation," and why does it happen as a synapse's peak approaches its reversal potential?
10. In the rate-coded transfer function, does output firing rate scale linearly with input rate, or does it saturate? Why?
11. What specific biological phenomenon would be lost if a two-compartment (soma + dendrite) model were collapsed into a single point neuron?

### D. Precision and Location in Circuit Design
12. Why does *timing*, not just synaptic strength, determine whether inhibition can suppress one specific spike without affecting neighboring spikes?
13. Does inhibition on the same dendrite as excitation need a smaller or larger weight to achieve the same suppression, compared to inhibition on a different dendrite?
14. Why aren't excitatory and inhibitory synapses randomly distributed across a real dendritic tree?
15. In feedforward inhibition (E→I), why does a smaller $\Delta t$ between excitation and inhibition onset produce a more strongly "clamped" EPSP peak?

### E. Modeling Practice
16. Why is it essential for an educational synapse model to keep parameters like $E_{rev}$ and $\tau$ in real biophysical units rather than arbitrary "strength" scores?
17. If a student's simulated EPSP overshoots the synapse's own reversal potential, what modeling assumption has been violated?
18. Why might a real circuit's simultaneous use of feedforward inhibition and spatial filtering be harder to interpret than studying each in isolation?

### 🎯 Your Turn — Design Two Questions, Not Just Answer Them
Pick any concept from this Set. Create **two** short questions of your own — one of each type:

**1. An Analysis-type question:** present a scenario, plot, or result, and ask the reader to interpret *why* something happened or what it implies.

**2. A Design-type question:** ask the reader *how* to achieve a specific target outcome, or what change would produce a given effect.

For each: include a brief setup, 2–4 possible answers, the correct answer, and a short explanation of why. No required format or topic beyond that — design both however makes sense to you. If your questions would genuinely catch someone who only skimmed this Set, you've demonstrated real understanding.

*Your Analysis-type question, answers, and explanation here:*

*Your Design-type question, answers, and explanation here:*

## The Modeler's Checklist (Final Reflection)
*Review these 10 points before submitting your notebook. These define a reproducible biophysical model.*

1. **Reproducibility:** Does your E0 block contain a fixed seed or `h.finitialize(-65)` to ensure your peer gets the same results?
2. **Transparency:** Are all your synaptic weights and time constants explicitly visible in your [Variable Key](#scrollTo=e0c8ab86) or sliders?
3. **Validation:** For every new circuit motif (like the E7 clamp), did you include a "Control" trace (Excitation only) for comparison?
4. **Unit Consistency:** Are your "Area" metrics clearly defined (e.g., $mV \cdot ms$)?
5. **Boundary Conditions:** Did you test what happens to your synapses when the membrane is at the reversal potential?
6. **Saturation:** Does your model account for the fact that driving force decreases as voltage approaches $E_{rev}$?
7. **Spatial Awareness:** Can you justify your placement of synapses (Proximal vs. Distal) based on the cell type you are modeling?
8. **Signal-to-Noise:** If you added noise, is the underlying "ground truth" signal still identifiable?
9. **Metric Selection:** Why did you choose *Area* over *Peak* to describe inhibitory strength in E8?
10. **Assumptions:** List one thing your model **omits** (e.g., active dendritic channels or $Ca^{2+}$ dynamics) and justify why that omission is acceptable for a beginner module.